<a href="https://colab.research.google.com/github/TheFaridKarami/farid-ai-lab/blob/main/projects/nlp/wnut17-robust-ner/notebooks/wnut17_multi_model_ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install -q seqeval

In [ ]:
import sys
import numpy as np
import torch, time
from torch import nn
import transformers
from transformers import (AutoTokenizer,
                          AutoModelForTokenClassification,
                          DataCollatorForTokenClassification,
                          TrainingArguments,
                          Trainer,
                          set_seed)

import datasets
from seqeval.metrics import classification_report, accuracy_score
from seqeval.scheme import IOB2

In [ ]:
wnut = datasets.load_dataset("flaitenberger/wnut_17")

In [ ]:
label_list = [label for label in wnut["train"].features["ner_tags"].feature.names]
id2label =   {i:label for i, label in enumerate(label_list)}
label2id =   {label:i for i, label in id2label.items()}


In [ ]:
shown = 0
for ex in wnut["train"]:

  if any(t != 0 for t in ex["ner_tags"]):
    print(" ".join(ex["tokens"]))
    print([(tok, id2label[t]) for tok, t in zip(ex["tokens"], ex["ner_tags"])])
    print()
  shown += 1

  if shown == 5:
    break

In [ ]:
for split in wnut:
  lens = [len(ex["tokens"]) for ex in wnut[split]]
  print(split, "mean:",round(np.mean(lens),2), "max: ", np.max(lens), "median: ", np.median(lens), "95th pct: ", np.quantile(lens, 0.95))

In [ ]:
from collections import Counter

for split in wnut:
  c = Counter(id2label[t] for ex in wnut[split] for t in ex["ner_tags"])
  ents = {l[2:]: n for l , n in c.items() if l.startswith("B-")}
  total = sum(c.values())
  print(split, "tokens:", total, "O share:", round(c["O"] / total, 3))
  print(ents, "total entities:", sum(ents.values()))

In [ ]:
tok = AutoTokenizer.from_pretrained("bert-base-cased")

words = ["Pxleyes", "Top", "50", "Photography", "Contest", "#photography"]

enc = tok(words, is_split_into_words= True)

print(tok.convert_ids_to_tokens(enc["input_ids"]))
print(enc.word_ids())

In [ ]:
def tokenize_and_align(batch):

  enc = tok(batch["tokens"], is_split_into_words=True, truncation=True, max_length=256)
  all_labels = []
  for i, tags in enumerate(batch["ner_tags"]):
    prev = None
    labs = []
    for w in enc.word_ids(batch_index=i):
      if w is None:
        labs.append(-100)
      elif w != prev:
        labs.append(tags[w])
      else:
        labs.append(-100)
      prev = w

    all_labels.append(labs)
  enc["labels"] = all_labels

  return enc

In [ ]:
tokenized = wnut.map(tokenize_and_align, batched = True, remove_columns = wnut["train"].column_names)

for split in tokenized:
  lost = sum(len(ex["tokens"]) - sum(l != -100 for l in lab) for ex, lab in zip(wnut[split], tokenized[split]["labels"]))
  print(split, "max_tokens:", max(len(x) for x in tokenized[split]["input_ids"]), "words lost:", lost)

In [ ]:
val_gold = [[id2label[t] for t in ex["ner_tags"]] for ex in wnut["validation"]]
test_gold = [[id2label[t] for t in ex["ner_tags"]] for ex in wnut["test"]]

In [ ]:
model = AutoModelForTokenClassification.from_pretrained("bert-base-cased", num_labels=len(label_list), id2label=id2label, label2id=label2id)
print(model.classifier)
print(sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad))

collator = DataCollatorForTokenClassification(tok)
batch = collator([tokenized["train"][i] for i in range(3)])
print({k: v.shape for k, v in batch.items()})
print(batch["labels"][0])

In [ ]:
def evaluate_ner(gold, pred):

    rep = classification_report(
        gold,
        pred,
        mode="strict",
        scheme=IOB2,
        output_dict=True,
        zero_division=0)

    out = {"precision": rep["micro avg"]["precision"],
           "recall": rep["micro avg"]["recall"],
           "f1": rep["micro avg"]["f1-score"],
           "token_acc": accuracy_score(gold, pred)}

    out["per_type_f1"] = {k: v["f1-score"] for k, v in rep.items() if "avg" not in k}

    return out

In [ ]:
def to_tags(logits, labels):

  preds = np.argmax(logits, axis = -1)
  gold, pred = [], []
  for pr, lb in zip(preds, labels):
    gold.append([id2label[l] for l in lb if l != -100])
    pred.append([id2label[q] for q, l in zip(pr, lb) if l != -100])

  return gold, pred

def compute_metrics(p):
  gold, pred = to_tags(*p)
  r = evaluate_ner(gold, pred)
  return {
      "precision":r["precision"],
      "f1":r["f1"],
      "recall": r["recall"],
      "accuracy": r["token_acc"]}




In [ ]:
b = collator([tokenized["validation"][i] for i in range(64)])
labels = b["labels"].numpy()
logits = np.eye(len(label_list))[np.where(labels == -100, 0, labels)]
print(compute_metrics((logits, labels)))

In [16]:
import torch, time
from transformers import Trainer, TrainingArguments, set_seed

set_seed(42)

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels = len(label_list),
    id2label = id2label,
    label2id = label2id)

args = TrainingArguments(
    output_dir="bert_base_cased_ckpt",
    learning_rate = 5e-5,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 64,
    num_train_epochs = 10,
    weight_decay = 0.01,
    warmup_steps = 107,
    fp16  =True,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    save_total_limit = 1,
    load_best_model_at_end = True,
    metric_for_best_model = "f1",
    seed = 42,
    report_to = "none")


trainer = Trainer(model = model,
                  args = args,
                  train_dataset = tokenized["train"],
                  eval_dataset  = tokenized["validation"],
                  data_collator = collator,
                  processing_class = tok,
                  compute_metrics = compute_metrics)

torch.cuda.reset_peak_memory_stats()
t0 = time.time()

trainer.train()


train_time = time.time() - t0
peak_gb = torch.cuda.max_memory_allocated() / 1e9
print("train time (s):", round(train_time), "peak GPU memory (GB):", round(peak_gb, 2))

Epoch,Training Loss,Validation Loss,Precision,F1,Recall,Accuracy
1,No log,0.243535,0.640232,0.489283,0.395933,0.944194
2,No log,0.215693,0.676692,0.526316,0.430622,0.949024
3,No log,0.232288,0.672161,0.531114,0.438995,0.951058
4,No log,0.229611,0.639517,0.565710,0.507177,0.952838
5,0.185357,0.265698,0.630603,0.550236,0.488038,0.952012
6,0.185357,0.275898,0.665625,0.577236,0.509569,0.953791
7,0.185357,0.313522,0.652648,0.566982,0.501196,0.953029
8,0.185357,0.351152,0.673504,0.554539,0.471292,0.952647
9,0.185357,0.337716,0.665079,0.571623,0.501196,0.953474
10,0.004516,0.352923,0.655172,0.552249,0.477273,0.952457


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

train time (s): 452 peak GPU memory (GB): 2.74


In [17]:
out = trainer.predict(tokenized["validation"])
gold, pred = to_tags(out.predictions, out.label_ids)
print(evaluate_ner(gold, pred))
print(gold == val_gold)
print(trainer.state.best_metric, trainer.state.best_model_checkpoint)

{'precision': np.float64(0.665625), 'recall': np.float64(0.5095693779904307), 'f1': np.float64(0.5772357723577236), 'token_acc': 0.953854954554122, 'per_type_f1': {'corporation': np.float64(0.1724137931034483), 'creative-work': np.float64(0.2916666666666667), 'group': np.float64(0.16216216216216214), 'location': np.float64(0.6119402985074627), 'person': np.float64(0.7258064516129032), 'product': np.float64(0.3838383838383838)}}
True
0.5772357723577236 bert_base_cased_ckpt/checkpoint-642


#**Architecture: RoBERTa-base**

In [35]:
tok_ro = AutoTokenizer.from_pretrained("FacebookAI/roberta-base", add_prefix_space = True)
enc = tok_ro(words, is_split_into_words= True)

In [44]:
print(enc.word_ids(batch_index=0))

[None, 0, 0, 0, 0, 1, 2, 3, 4, 5, 5, 5, None]


In [49]:
def tokenize_and_align(batch):

 enc = tok_ro(batch["train"], is_split_into_words = True, max_length = 512, truncation= True)

 all_labels = []

 for i, tags in enumerate(batch["ner_tags"]):
  prev = None
  labs = []

  for w in enc.word_ids(batch_index = i):

    if w == None:
      labs.append(-100)
    elif w != prev:
      labs.append(tags[w])
    else:
      labs.append(-100)
    prev = w

  all_labels.append(labs)

  enc["labels"] = all_labels

  return enc

In [51]:
tokenize = wnut.map(
    tokenize_and_align,
    batched =True,
    remove_columns = wnut["train"].column_names,
    fn_kwargs = {"tokenizer": tok})

Map:   0%|          | 0/3394 [00:00<?, ? examples/s]

TypeError: tokenize_and_align() got an unexpected keyword argument 'tokenizer'